# Phase 2 — Text Cleaning & Preprocessing

Loads cleaned data from Phase 1 (`master_data` in PostgreSQL or CSV), applies NLP preprocessing, and saves **`cleaned_reviews_phase2.csv`**.

**Phase 1 schema** (single table `master_data`):
`row_id`, `target`, `ids`, `date`, `flag`, `user`, `text` (+ any custom columns from your CSV header)

```bash
pip install pandas sqlalchemy psycopg2-binary nltk spacy
python -m spacy download en_core_web_sm
python -c "import nltk; nltk.download('punkt'); nltk.download('stopwords')"
```

## Install dependencies (run once)

Run if you see `ModuleNotFoundError: No module named 'spacy'`.

In [1]:
%pip install -q spacy nltk
%pip install -q https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Setup — Imports and configuration

In [2]:
from __future__ import annotations

import re
import sys
import time
from pathlib import Path
from typing import Iterable, List, Optional, Set

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine
from sqlalchemy.exc import SQLAlchemyError

PROJECT_DIR = Path.cwd().resolve()
OUTPUT_CSV = PROJECT_DIR / "cleaned_reviews_phase2.csv"
MISSING_REVIEW_PLACEHOLDER = "missing_review"
MASTER_TABLE_NAME = "master_data"  # Phase 1 PostgreSQL table
LEMMA_BATCH_SIZE = 2000

REVIEW_TEXT_CANDIDATES = [
    "text", "review_text", "review", "comment", "body",
    "content", "description", "message", "tweet",
]

print(f"Project directory: {PROJECT_DIR}")

Project directory: D:\NCPL\Bootcamp\Projects\Project 5\new data\data 2


---
## Step 1 — Load data

- **postgres** — reads Phase 1 table **`master_data`** (default)
- **csv** — reads a file from the project folder

Review text column is auto-detected (`text` for the Twitter/sentiment dataset).

In [3]:
def find_review_text_column(columns: List[str]) -> str:
    normalized = {c: re.sub(r"[^a-z0-9]", "", c.lower()) for c in columns}
    for candidate in REVIEW_TEXT_CANDIDATES:
        key = re.sub(r"[^a-z0-9]", "", candidate)
        for col, norm in normalized.items():
            if norm == key or key in norm or norm in key:
                return col
    for col in columns:
        if "text" in col.lower() or "review" in col.lower():
            return col
    raise ValueError(f"No review text column found. Columns: {columns}")


def prompt_db_credentials() -> dict:
    return {
        "host": input("PostgreSQL host: ").strip(),
        "port": input("PostgreSQL port (5432): ").strip() or "5432",
        "database": input("Database name: ").strip(),
        "username": input("Username: ").strip(),
        "password": input("Password: ").strip(),
    }


def create_pg_engine(creds: dict) -> Engine:
    missing = [k for k in ("host", "database", "username", "password") if not creds.get(k)]
    if missing:
        raise ValueError(f"Missing connection fields: {missing}")
    url = (
        f"postgresql+psycopg2://{creds['username']}:{creds['password']}"
        f"@{creds['host']}:{creds['port']}/{creds['database']}"
    )
    return create_engine(url, pool_pre_ping=True)


def load_from_postgres(engine: Engine, table: str = MASTER_TABLE_NAME) -> pd.DataFrame:
    try:
        df = pd.read_sql(text(f"SELECT * FROM {table}"), engine)
    except SQLAlchemyError as exc:
        raise RuntimeError(f"Failed to read '{table}': {exc}") from exc
    if df.empty:
        raise ValueError(f"Table '{table}' is empty. Run Phase 1 first.")
    return df


def load_from_csv(project_dir: Path) -> pd.DataFrame:
    csv_name = input("Enter CSV file name: ").strip()
    path = project_dir / csv_name
    if not path.is_file():
        raise FileNotFoundError(f"CSV not found: {path}")
    try:
        return pd.read_csv(path, encoding="utf-8", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", low_memory=False)


def load_dataset(project_dir: Path) -> tuple[pd.DataFrame, str]:
    source = input("Load from 'postgres' or 'csv'? ").strip().lower()
    if source == "postgres":
        engine = create_pg_engine(prompt_db_credentials())
        default = MASTER_TABLE_NAME
        table = input(f"Table name (default: {default}): ").strip() or default
        df = load_from_postgres(engine, table)
    elif source == "csv":
        df = load_from_csv(project_dir)
    else:
        raise ValueError("Source must be 'postgres' or 'csv'.")

    text_col = find_review_text_column(list(df.columns))
    df[text_col] = df[text_col].astype("string")
    null_count = int(df[text_col].isna().sum())
    print(f"Loaded {len(df):,} rows from {'PostgreSQL' if source == 'postgres' else 'CSV'}")
    print(f"Review column: '{text_col}' | Null text on load: {null_count:,}")
    print(f"Columns: {list(df.columns)}")
    return df, text_col


try:
    df, TEXT_COL = load_dataset(PROJECT_DIR)
    display(df.head(3))
except (ValueError, FileNotFoundError, RuntimeError, SQLAlchemyError) as exc:
    print(f"Load error: {exc}", file=sys.stderr)
    raise

Loaded 1,581,466 rows from PostgreSQL
Review column: 'text' | Null text on load: 0
Columns: ['row_id', 'target', 'ids', 'date', 'flag', 'user', 'text']


,row_id,target,ids,date,flag,user,text
0,1,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,2,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,3,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...


---
## Reusable cleaning functions

`clean_text()`, `remove_stopwords()`, `tokenize_text()`, `lemmatize_series_fast()`

In [4]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
EMOJI_PATTERN = re.compile(
    "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF\U00002702-\U000027B0\U000024C2-\U0001F251]+",
    flags=re.UNICODE,
)
NON_ALNUM_SPACE = re.compile(r"[^a-z0-9\s]+")
WHITESPACE = re.compile(r"\s+")
_SPACY_NLP = None


def clean_text(
    text: str,
    *,
    lowercase: bool = True,
    remove_urls: bool = True,
    remove_special: bool = True,
) -> str:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""
    out = str(text).strip()
    if not out or out.lower() in ("nan", "none", "null"):
        return ""
    if lowercase:
        out = out.lower()
    if remove_urls:
        out = URL_PATTERN.sub("", out)
    if remove_special:
        out = EMOJI_PATTERN.sub("", out)
        out = NON_ALNUM_SPACE.sub(" ", out)
        out = WHITESPACE.sub(" ", out).strip()
    return out


def _ensure_nltk_resources() -> None:
    import nltk
    for path, pkg in [("corpora/stopwords", "stopwords"), ("tokenizers/punkt", "punkt")]:
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(pkg, quiet=True)


def _load_spacy_model():
    global _SPACY_NLP
    if _SPACY_NLP is not None:
        return _SPACY_NLP
    try:
        import spacy
    except ModuleNotFoundError as exc:
        raise RuntimeError("spaCy not installed. Run the Install dependencies cell.") from exc
    try:
        _SPACY_NLP = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    except OSError as exc:
        raise RuntimeError("Run: python -m spacy download en_core_web_sm") from exc
    return _SPACY_NLP


def get_stopword_set(custom: Optional[Iterable[str]] = None, backend: str = "nltk") -> Set[str]:
    custom_set = {w.strip().lower() for w in (custom or []) if w and str(w).strip()}
    if backend == "spacy":
        base = _load_spacy_model().Defaults.stop_words
    else:
        _ensure_nltk_resources()
        from nltk.corpus import stopwords
        base = set(stopwords.words("english"))
    return set(base) | custom_set


def remove_stopwords(text: str, stop_words: Set[str]) -> str:
    if not text or not str(text).strip():
        return ""
    return " ".join(t for t in str(text).split() if t.lower() not in stop_words)


def tokenize_text(text: str, backend: str = "nltk", nlp=None) -> List[str]:
    if not text or not str(text).strip():
        return []
    if backend == "spacy":
        nlp = nlp or _load_spacy_model()
        return [t.text for t in nlp(str(text)) if t.text.strip()]
    _ensure_nltk_resources()
    from nltk.tokenize import word_tokenize
    return [t for t in word_tokenize(str(text)) if t.strip()]


def tokenize_series_fast(texts: pd.Series, nlp=None, batch_size: int = LEMMA_BATCH_SIZE) -> pd.Series:
    nlp = nlp or _load_spacy_model()
    out = {idx: [] for idx in texts.index}
    work_idx, work_text = [], []
    for idx, val in texts.items():
        s = str(val).strip() if pd.notna(val) else ""
        if s:
            work_idx.append(idx)
            work_text.append(s)
    for doc, idx in zip(nlp.pipe(work_text, batch_size=batch_size), work_idx):
        out[idx] = [t.text for t in doc if t.text.strip()]
    return pd.Series([out[i] for i in texts.index], index=texts.index)


def _doc_to_lemmas(doc) -> List[str]:
    return [
        tok.lemma_.lower().strip()
        for tok in doc
        if tok.lemma_.strip() and tok.lemma_.lower() != "-pron-"
    ]


def lemmatize_text(tokens: List[str], nlp=None) -> List[str]:
    if not tokens:
        return []
    nlp = nlp or _load_spacy_model()
    return _doc_to_lemmas(nlp(" ".join(str(t) for t in tokens)))


def lemmatize_series_fast(
    texts: pd.Series,
    nlp=None,
    batch_size: int = LEMMA_BATCH_SIZE,
) -> pd.Series:
    nlp = nlp or _load_spacy_model()
    out = {idx: [] for idx in texts.index}
    work_idx, work_text = [], []
    for idx, val in texts.items():
        s = str(val).strip() if pd.notna(val) else ""
        if s:
            work_idx.append(idx)
            work_text.append(s)
    for doc, idx in zip(nlp.pipe(work_text, batch_size=batch_size), work_idx):
        out[idx] = _doc_to_lemmas(doc)
    return pd.Series([out[i] for i in texts.index], index=texts.index)

---
## Step 2 — Lowercase conversion

Normalizes casing so tokens match during search and deduplication.

In [5]:
df["text_raw"] = df[TEXT_COL].astype("string")
df["text_lower"] = df["text_raw"].apply(
    lambda x: clean_text(x, lowercase=True, remove_urls=False, remove_special=False)
)
display(df[[TEXT_COL, "text_lower"]].head(3))

,text,text_lower
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...","@switchfoot http://twitpic.com/2y1zl - awww, t..."
1,is upset that he can't update his Facebook by ...,is upset that he can't update his facebook by ...
2,@Kenichan I dived many times for the ball. Man...,@kenichan i dived many times for the ball. man...


---
## Step 3 — Remove URLs

Strips `http://`, `https://`, and `www.` links via regex.

In [6]:
df["text_no_url"] = df["text_lower"].apply(
    lambda x: clean_text(x, lowercase=False, remove_urls=True, remove_special=False)
)
display(df[["text_lower", "text_no_url"]].head(3))

,text_lower,text_no_url
0,"@switchfoot http://twitpic.com/2y1zl - awww, t...","@switchfoot - awww, that's a bummer. you sho..."
1,is upset that he can't update his facebook by ...,is upset that he can't update his facebook by ...
2,@kenichan i dived many times for the ball. man...,@kenichan i dived many times for the ball. man...


---
## Step 4 — Remove emojis and special characters

Full `clean_text()` — alphanumeric and spaces only.

In [7]:
df["text_clean"] = df[TEXT_COL].apply(clean_text)
display(df[[TEXT_COL, "text_clean"]].head(3))

,text,text_clean
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot awww that s a bummer you shoulda go...
1,is upset that he can't update his Facebook by ...,is upset that he can t update his facebook by ...
2,@Kenichan I dived many times for the ball. Man...,kenichan i dived many times for the ball manag...


---
## Step 5 — Remove stopwords

NLTK English stopwords by default; add custom words at the prompt.

In [8]:
STOP_BACKEND = "nltk"
custom_raw = input("Custom stopwords (comma-separated, or Enter to skip): ").strip()
custom_stopwords = [w.strip() for w in custom_raw.split(",") if w.strip()] if custom_raw else []

STOP_WORDS = get_stopword_set(custom_stopwords, backend=STOP_BACKEND)
df["text_no_stopwords"] = df["text_clean"].apply(lambda x: remove_stopwords(x, STOP_WORDS))
print(f"Stopword count: {len(STOP_WORDS):,}")
display(df[["text_clean", "text_no_stopwords"]].head(3))

Stopword count: 198


,text_clean,text_no_stopwords
0,switchfoot awww that s a bummer you shoulda go...,switchfoot awww bummer shoulda got david carr ...
1,is upset that he can t update his facebook by ...,upset update facebook texting might cry result...
2,kenichan i dived many times for the ball manag...,kenichan dived many times ball managed save 50...


---
## Step 6 — Tokenization

NLTK `word_tokenize` by default; set `TOKEN_BACKEND = "spacy"` for spaCy.

In [9]:
TOKEN_BACKEND = "nltk"

if TOKEN_BACKEND == "spacy":
    nlp_tok = _load_spacy_model()
    df["tokens_before_stop"] = tokenize_series_fast(df["text_clean"], nlp_tok)
    df["tokens"] = tokenize_series_fast(df["text_no_stopwords"], nlp_tok)
else:
    df["tokens_before_stop"] = df["text_clean"].apply(
        lambda x: tokenize_text(x, backend="nltk")
    )
    df["tokens"] = df["text_no_stopwords"].apply(
        lambda x: tokenize_text(x, backend="nltk")
    )

print("Sample tokens (before / after stopwords):")
print(f"  BEFORE: {df.loc[0, 'tokens_before_stop'][:12]}")
print(f"  AFTER:  {df.loc[0, 'tokens'][:12]}")

Sample tokens (before / after stopwords):
  BEFORE: ['switchfoot', 'awww', 'that', 's', 'a', 'bummer', 'you', 'shoulda', 'got', 'david', 'carr', 'of']
  AFTER:  ['switchfoot', 'awww', 'bummer', 'shoulda', 'got', 'david', 'carr', 'third', 'day']


---
## Step 7 — Lemmatization

spaCy batch lemmatization via `nlp.pipe` (fast on large datasets).

In [10]:
nlp = _load_spacy_model()
batch_size = globals().get("LEMMA_BATCH_SIZE", 2000)

print(f"Lemmatizing {len(df):,} rows (batch_size={batch_size})...")
t0 = time.perf_counter()
df["lemmas"] = lemmatize_series_fast(df["text_no_stopwords"], nlp=nlp, batch_size=batch_size)
df["text_lemmatized"] = df["lemmas"].apply(lambda xs: " ".join(xs) if xs else "")
print(f"Done in {time.perf_counter() - t0:.1f}s")

print(f"  tokens: {df.loc[0, 'tokens'][:10]}")
print(f"  lemmas: {df.loc[0, 'lemmas'][:10]}")

Lemmatizing 1,581,466 rows (batch_size=2000)...
Done in 1439.6s
  tokens: ['switchfoot', 'awww', 'bummer', 'shoulda', 'got', 'david', 'carr', 'third', 'day']
  lemmas: ['switchfoot', 'awww', 'bummer', 'shoulda', 'get', 'david', 'carr', 'third', 'day']


---
## Step 8 — Handle missing review text

Fill null/empty text with `missing_review`; drop rows still unusable after cleaning.

In [11]:
def is_missing_text(val) -> bool:
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return True
    s = str(val).strip().lower()
    return s in ("", "nan", "none", "null", "<na>")


missing_mask = df[TEXT_COL].apply(is_missing_text) | df["text_clean"].apply(is_missing_text)
missing_handled = int(missing_mask.sum())

for col in (TEXT_COL, "text_clean", "text_no_stopwords", "text_lemmatized"):
    df.loc[missing_mask, col] = MISSING_REVIEW_PLACEHOLDER

before_drop = len(df)
unusable = df["text_clean"].apply(is_missing_text) & df["text_lemmatized"].apply(is_missing_text)
df = df.loc[~unusable].reset_index(drop=True)
rows_dropped = before_drop - len(df)

print(f"Missing filled with '{MISSING_REVIEW_PLACEHOLDER}': {missing_handled:,}")
print(f"Rows dropped (unusable): {rows_dropped:,}")
print(f"Rows remaining: {len(df):,}")

Missing filled with 'missing_review': 21
Rows dropped (unusable): 0
Rows remaining: 1,581,466


---
## Step 9 — Remove duplicate reviews

Dedup on **`text_clean`** (Phase 1 already deduped raw `text`; this catches post-cleaning duplicates).

In [12]:
before_dedup = len(df)
df = df.drop_duplicates(subset=["text_clean"], keep="first").reset_index(drop=True)
duplicates_removed_phase2 = before_dedup - len(df)

print(f"Before: {before_dedup:,} | After: {len(df):,} | Removed: {duplicates_removed_phase2:,}")

Before: 1,581,466 | After: 1,565,849 | Removed: 15,617


---
## Step 10 — Final output

Saves all original `master_data` columns plus engineered text fields to **`cleaned_reviews_phase2.csv`**.

In [13]:
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print("=" * 60)
print("PHASE 2 — PIPELINE SUMMARY")
print("=" * 60)
print(f"Source table (Phase 1): {MASTER_TABLE_NAME}")
print(f"Output file:            {OUTPUT_CSV}")
print(f"Missing values handled: {missing_handled:,}")
print(f"Duplicates removed:     {duplicates_removed_phase2:,}")
print(f"Final row count:        {len(df):,}")
print(f"\nColumns saved: {list(df.columns)}")
display(df[[TEXT_COL, "text_clean", "text_lemmatized"]].head(5))
print(f"\nToken sample:  {df.loc[0, 'tokens'][:15]}")
print(f"Lemma sample:  {df.loc[0, 'lemmas'][:15]}")
print("\nPhase 2 complete.")

PHASE 2 — PIPELINE SUMMARY
Source table (Phase 1): master_data
Output file:            D:\NCPL\Bootcamp\Projects\Project 5\new data\data 2\cleaned_reviews_phase2.csv
Missing values handled: 21
Duplicates removed:     15,617
Final row count:        1,565,849

Columns saved: ['row_id', 'target', 'ids', 'date', 'flag', 'user', 'text', 'text_raw', 'text_lower', 'text_no_url', 'text_clean', 'text_no_stopwords', 'tokens_before_stop', 'tokens', 'lemmas', 'text_lemmatized']


,text,text_clean,text_lemmatized
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot awww that s a bummer you shoulda go...,switchfoot awww bummer shoulda get david carr ...
1,is upset that he can't update his Facebook by ...,is upset that he can t update his facebook by ...,upset update facebook texting might cry result...
2,@Kenichan I dived many times for the ball. Man...,kenichan i dived many times for the ball manag...,kenichan dive many time ball manage save 50 re...
3,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire,whole body feel itchy like fire
4,"@nationwideclass no, it's not behaving at all....",nationwideclass no it s not behaving at all i ...,nationwideclass behave mad see



Token sample:  ['switchfoot', 'awww', 'bummer', 'shoulda', 'got', 'david', 'carr', 'third', 'day']
Lemma sample:  ['switchfoot', 'awww', 'bummer', 'shoulda', 'get', 'david', 'carr', 'third', 'day']

Phase 2 complete.
